# 1. 利用KNN算法实现车型推荐



## 0. 载入相关模块

In [8]:
import pandas as pd
import sklearn.neighbors as neighbors

## 1. 数据预处理

### 1.1 载入数据

In [18]:
df = pd.read_csv("原始-扩展数据.csv", encoding="gbk", )
df

,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元)
0,BAOJ310-,4032,1680,1450,5.3,5.6
1,XW-M1+,4330,1535,1885,7.8,14.5
2,POL-,4053,1740,1449,6.2,10.8
3,BW5+,5087,1868,1500,8.5,25.6
4,BDe6+,4560,1822,1645,7.8,15.8
5,Wuling-,3797,1510,1820,5.5,9.6
6,CRV,4585,1855,1679,7.6,19.6


### 1.2 数据预处理

1. 数据归一化
2. 区分已知数据和待预测数据
3. 划分训练集和测试集

In [22]:
feature_cols = df.columns[1:]  # 特征列
regular_df = df.copy()
regular_df[feature_cols] = regular_df[feature_cols].apply(
    lambda col: (col - col.min()) / (col.max() - col.min())
)

regular_df 

,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元)
0,BAOJ310-,0.182171,0.474860,0.002294,0.00000,0.000
1,XW-M1+,0.413178,0.069832,1.000000,0.78125,0.445
2,POL-,0.198450,0.642458,0.000000,0.28125,0.260
3,BW5+,1.000000,1.000000,0.116972,1.00000,1.000
4,BDe6+,0.591473,0.871508,0.449541,0.78125,0.510
5,Wuling-,0.000000,0.000000,0.850917,0.06250,0.200
6,CRV,0.610853,0.963687,0.527523,0.71875,0.700


In [23]:
# (1) 创建新列“推荐”，用于存储推荐结果
regular_df["推荐"] = None

# (2) 获取所有型号的推荐信息
# 已知数据：型号后缀为+或-；+ 代表推荐，- 代表不推荐
# 未知数据：型号后缀为空，代表未知

for name in regular_df["型号"].values:
    if name[-1] == "+":
        regular_df.loc[regular_df["型号"] == name, "推荐"] = 1
    elif name[-1] == "-":
        regular_df.loc[regular_df["型号"] == name, "推荐"] = 0

unknown_df = regular_df[regular_df["推荐"].isnull()]
known_df = regular_df[~regular_df["推荐"].isnull()]
# 指定known_df推荐列为整数类型
known_df["推荐"] = known_df["推荐"].astype(int)

display(unknown_df)
display(known_df)

,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元),推荐
6,CRV,0.610853,0.963687,0.527523,0.71875,0.7,None


,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元),推荐
0,BAOJ310-,0.182171,0.474860,0.002294,0.00000,0.000,0
1,XW-M1+,0.413178,0.069832,1.000000,0.78125,0.445,1
2,POL-,0.198450,0.642458,0.000000,0.28125,0.260,0
3,BW5+,1.000000,1.000000,0.116972,1.00000,1.000,1
4,BDe6+,0.591473,0.871508,0.449541,0.78125,0.510,1
5,Wuling-,0.000000,0.000000,0.850917,0.06250,0.200,0


In [25]:
# (3) 训练数据
train_df = known_df.iloc[:4, :]
test_df = known_df.iloc[4:, :]

display(train_df)
display(test_df)

,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元),推荐
0,BAOJ310-,0.182171,0.474860,0.002294,0.00000,0.000,0
1,XW-M1+,0.413178,0.069832,1.000000,0.78125,0.445,1
2,POL-,0.198450,0.642458,0.000000,0.28125,0.260,0
3,BW5+,1.000000,1.000000,0.116972,1.00000,1.000,1


,型号,长(mm),宽(mm),高(mm),油耗(L/100km),售价(万元),推荐
4,BDe6+,0.591473,0.871508,0.449541,0.78125,0.51,1
5,Wuling-,0.000000,0.000000,0.850917,0.06250,0.20,0


## 2. 模型训练

（1）创建模型

（2）模型训练

In [27]:
# (1) 创建模型
knn = neighbors.KNeighborsClassifier(n_neighbors=3, weights='distance')

# (2) 模型训练
knn.fit(train_df.iloc[:, 1:-1].values, train_df.iloc[:, -1].values)

# (3) 模型预测
test_df_pre = knn.predict(test_df.iloc[:, 1:-1].values)

# (4) 模型评估
for i in range(len(test_df_pre)):
    print("型号：%s，实际推荐：%d，预测推荐：%d" % (test_df.iloc[i, 0], test_df.iloc[i, -1], test_df_pre[i]))

scores = knn.score(test_df.iloc[:, 1:-1].values, test_df.iloc[:, -1].values)
print("模型准确率：%.2f%%" % (scores * 100))

型号：BDe6+，实际推荐：1，预测推荐：1
型号：Wuling-，实际推荐：0，预测推荐：0
模型准确率：100.00%


## 3. 模型保存与使用

In [15]:
# # (1) 保存模型
# import joblib
# joblib.dump(knn, "knn_model.pkl")

# # (2) 加载模型
# knn_model = joblib.load("knn_model.pkl")

In [28]:
# 模型使用
results = knn.predict(unknown_df.iloc[:, 1:-1].values)
print("未知型号的推荐结果：")
for i in range(len(results)):
    print("型号：%s，预测推荐：%d" % (unknown_df.iloc[i, 0], results[i]))

未知型号的推荐结果：
型号：CRV，预测推荐：1
